In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.electiva.mi_volumen_datos;

In [0]:
%python
ruta = "/Volumes/workspace/electiva/mi_volumen_datos/datos (2) (1).zip"
display(dbutils.fs.ls(ruta))


In [0]:
%python
import zipfile
import os
ruta = "/Volumes/workspace/electiva/mi_volumen_datos/"
archivo_zip = ruta + "datos (2) (1).zip"
carpeta_destino = ruta + "csv_extraidos"
os.makedirs(carpeta_destino, exist_ok=True)
with zipfile.ZipFile(archivo_zip,"r") as zip_ref:
    zip_ref.extractall(carpeta_destino)
print("Extracción completada")

In [0]:
%python
display(dbutils.fs.ls(
    "/Volumes/workspace/electiva/mi_volumen_datos/csv_extraidos/"
))

In [0]:
%python
ruta_csv = "/Volumes/workspace/electiva/mi_volumen_datos/csv_extraidos/"

archivos = dbutils.fs.ls(ruta_csv)

for archivo in archivos:
    print(archivo.name)

CREAR TABLAS 


In [0]:
%python
# Ruta donde se encuentran los archivos CSV
ruta="/Volumes/workspace/electiva/mi_volumen_datos/csv_extraidos/"

# Lista de archivos y nombres de las tablas
tablas={"customers.csv":"clientes",
        "employees.csv":"empleados",
        "orders.csv":"ordenes",
        "productcategories.csv":"categorias_de_productos",
        "productsubcategories.csv":"subcategorias_de_productos",
        "products.csv":"productos",
        "vendors.csv":"vendedores",
        "vendorproduct.csv":"vendedores_de_productos"}

# Recorrer los archivos y crear las tablas Delta
for archivo,nombre_tabla in tablas.items():
    df=spark.read\
        .option("header", True)\
        .option("inferSchema",True)\
        .csv(ruta+archivo)
    df.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"workspace.electiva.{nombre_tabla}")
    print(f"tabla {nombre_tabla} creada con éxito")
       


 PUNTO 1: Desarrollar una consulta SQL para obtener una tabla con el customerid, el fullname y el user_name de la persona.

In [0]:
%sql

SELECT
CustomerID,
concat(FirstName, LastName) as fullname,
concat(
    customerID, "-",
    upper(substring(FirstName, 1, 2)), "-",
    lower(substring(LastName, 1, 2)), "-AW"
) As user_name
From workspace.electiva.customers;


PUNTO2: cree una tabla 
llamada page_user que contenga dos columnas: el customerid y el user_name e inserte
esta información en la tabla recién creada para ser consultada directamente.

In [0]:
%sql
CREATE TABLE if not exists workspace.electiva.page_user(
    customerID  STRING,
    user_name STRING) ;

    




In [0]:
%sql
INSERT INTO workspace.electiva.page_user
SELECT
CustomerID,
CONCAT(CAST(CustomerID AS string),"-",
      UPPER(SUBSTRING(Firstname,1,2)),"-",
      LOWER(SUBSTRING(LastName,1,2)),"-AW") AS user_name
      FROM workspace.electiva.customers

      LIMIT 100;


In [0]:
%sql
SELECT *
FROM workspace.electiva.page_user
LIMIT 100;

PUNTO3

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.electiva.ordenes_agosto_2011 AS

SELECT
p.productname,
p.modelname,
pu.user_name,
o.salesorderid,
cp.categoryname,
o.orderdate,
o.linetotal,
CASE

        WHEN cp.categoryname ILIKE 'Bikes' THEN 0.10
        WHEN cp.categoryname ILIKE 'Clothing' THEN 0.15
        WHEN cp.categoryname ILIKE 'Components' THEN 0.08
        WHEN cp.categoryname ILIKE 'Accessories' THEN 0.20
        ELSE 0
    END AS `descuento_mitad_año`,


o.linetotal - (
        o.linetotal *
        CASE
            WHEN cp.categoryname ILIKE 'Bikes' THEN 0.10
            WHEN cp.categoryname ILIKE 'Clothing' THEN 0.15
            WHEN cp.categoryname ILIKE 'Components' THEN 0.08
            WHEN cp.categoryname ILIKE 'Accessories' THEN 0.20
            ELSE 0
        END
    ) AS total_descuentos_aplicados

FROM workspace.electiva.ordenes o

INNER JOIN workspace.electiva.productos p
    ON o.productid = p.productid

INNER JOIN workspace.electiva.subcategorias_de_productos sp
    ON p.subcategoryid = sp.subcategoryid

INNER JOIN workspace.electiva.categorias_de_productos cp
    ON sp.categoryid = cp.categoryid

INNER JOIN workspace.electiva.page_user pu
    ON o.CustomerID = pu.CustomerID

WHERE o.orderdate >= '2011-08-01'
  AND o.orderdate < '2011-09-01';

SELECT
linetotal - (
        linetotal *
        CASE
            WHEN categoryname ILIKE 'Bikes' THEN 0.10
            WHEN categoryname ILIKE 'Clothing' THEN 0.15
            WHEN categoryname ILIKE 'Components' THEN 0.08
            WHEN categoryname ILIKE 'Accessories' THEN 0.20
            ELSE 0
        END
    ) AS total_descuentos_aplicados

FROM workspace.electiva.ordenes_agosto_2011;

In [0]:
%sql
SELECT *
FROM workspace.electiva.ordenes_agosto_2011;

**Nota aclaratoria:**

Al desarrollar el cálculo de la columna `total_descuentos_aplicados`, se utilizó la fórmula indicada en el enunciado:

`linetotal - (linetotal * descuento_mitad_año)`.

Sin embargo, se identificó una diferencia en el resultado de referencia presentado en la página 3 del documento. Para el registro con `linetotal = 4079.988` y un descuento de `0.10` (10%), el resultado esperado según la fórmula es `3671.9892`, mientras que en la imagen del profesor aparece `4079.888`.

Por lo tanto, se conservó la fórmula escrita en el enunciado, ya que representa correctamente la aplicación porcentual del descuento del 10%.

punto4

Crear una vista materializada en el esquema, llamada customer_class que contenga el
user_name de todos los clientes del año 2011 , una columna user_level donde indique si el
cliente es “Oro”, “Plata” o “Ocasional”, además debe contener para cada cliente su
customerid, su customer fullname y la cantidad de órdenes que realizó en el año.


In [0]:
%sql
SELECT
   CustomerID,
   COUNT(DISTINCT SalesOrderID) AS cantidad_ordenes
FROM workspace.electiva.ordenes
Where YEAR(orderdate) = 2011
    GROUP BY CustomerID




In [0]:
%sql
WITH conteo_clientes AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT SalesOrderID) AS cantidad_ordenes
    FROM workspace.electiva.ordenes
    WHERE YEAR(orderdate) = 2011
    GROUP BY CustomerID
)

SELECT
     pu.user_name,
CASE
 WHEN cc.cantidad_ordenes= 3 THEN "Oro"
 WHEN cc.cantidad_ordenes= 2 THEN "Plata"
 WHEN cc.cantidad_ordenes< 2 THEN "Ocasional"
END AS use_level,
cc.customerID,
CONCAT( c.firstname," ",c.lastname) as `full name`,
cc.cantidad_ordenes

FROM conteo_clientes cc
left join workspace.electiva.page_user pu
on cc.customerID=pu.customerid
left join workspace.electiva.customers c
on cc.customerID=c.customerid;






In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW workspace.electiva.customer_class AS

WITH conteo_clientes AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT SalesOrderID) AS cantidad_ordenes
    FROM workspace.electiva.ordenes
    WHERE YEAR(orderdate) = 2011
    GROUP BY CustomerID
)

SELECT
    pu.user_name,
    CASE
        WHEN cc.cantidad_ordenes = 3 THEN 'Oro'
        WHEN cc.cantidad_ordenes = 2 THEN 'Plata'
        WHEN cc.cantidad_ordenes < 2 THEN 'Ocasional'
    END AS user_level,
    cc.CustomerID,
    CONCAT(c.firstname, ' ', c.lastname) AS fullname,
    cc.cantidad_ordenes
FROM conteo_clientes cc
LEFT JOIN workspace.electiva.page_user pu
    ON cc.CustomerID = pu.CustomerID
LEFT JOIN workspace.electiva.customers c
    ON cc.CustomerID = c.customerid;